<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/tokenization/notebooks/03_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data preprocessing

Tokenization + BIO labeling

## Relevant EDA Findings:

- Character-level annotations: loc_markers stores location spans as [start, end) offsets. Tokenization must preserve alignment with the original text.

- Annotation edge cases: EDA identified occasional whitespace at span boundaries, incorrect boundaries and annotation noise. These cases require attention during BIO label assignment.

- Multiword locations: 15.39% of UK and 11.86% of ru annotated spans contain multiple words. BIO labels must preserve entity boundaries across tokens.

- Mixed-language text: Ukrainian, russian, Latin/Cyrillic mixtures and homoglyphs occur in the data. Tokenization should handle them without modifying the original character positions.

- Long texts: Test texts are substantially longer than training texts and frequently contain line breaks, emojis, URLs and hashtags. Tokenization and sequence-length handling must account for these cases.

- Data leakage prevention: Tokenizer vocabulary must be trained on the training split only, using the existing validation_manifest.parquet

## Environment Setup

In [1]:
import sys
import ast
import random
from pathlib import Path

import os
import base64
import subprocess

import numpy as np
import pandas as pd

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [2]:
if "google.colab" in sys.modules:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    root = "/content"
else:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    root = "/kaggle/working"

repo = "UA-Locations-NER-Deep-Learning"
branch = "tokenization"
repo_dir = f"{root}/{repo}"

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

if not os.path.exists(repo_dir):
    subprocess.run(
        ["git", "clone", "-b", branch, f"https://github.com/dkalchenko01/{repo}.git", repo_dir],
        env=env, check=True
    )

print("Repository:", repo_dir)

Repository: /content/UA-Locations-NER-Deep-Learning


In [4]:
sys.path.insert(0, repo_dir)

from src.data_io import get_data_dir, load_manifest, select_manifest_rows, load_data_splits
from src.preprocessing import preprocess_data
from src.eda_scan import parse_spans

splits_dir = Path(repo_dir) / "data" / "splits"

print("Modules imported successfully")
print("Validation manifest exists:", (splits_dir / "validation_manifest.parquet").exists())

Modules imported successfully
Validation manifest exists: True


## Data Loading

In [5]:
ENVIRONMENT = "colab" if root == "/content" else "kaggle"

data_dir = get_data_dir(ENVIRONMENT)

print("Data directory:", data_dir)
print("\nAvailable files:")

for path in sorted(data_dir.iterdir()):
    print(path.name)

Data directory: /root/.cache/kagglehub/competitions/kse-ua-location-extraction-2026

Available files:
README.md
labeling_sample.csv
ru_geo_dataset.csv
test.csv
uk_geo_dataset.csv
uk_geo_dataset_processed_v1.parquet


Train/val splits loading:

In [6]:
splits = load_data_splits(ENVIRONMENT, splits_dir)

for name, df in splits.items():
    print(f"{name}: {df.shape}, index unique: {df.index.is_unique}")

uk_train: (805983, 5), index unique: True
uk_val: (201498, 5), index unique: True
ru_train: (100000, 6), index unique: True
ru_val: (49369, 6), index unique: True


Check train/validation overlap

In [7]:
for source in ("uk", "ru"):
    train = splits[f"{source}_train"]
    val = splits[f"{source}_val"]

    overlap = train.index.intersection(val.index)

    print(f"{source.upper()}:")
    print(f"  Train: {len(train):,}")
    print(f"  Validation: {len(val):,}")
    print(f"  Overlapping row_ids: {len(overlap)}")

    assert len(overlap) == 0, f"{source}: train/val overlap detected"

UK:
  Train: 805,983
  Validation: 201,498
  Overlapping row_ids: 0
RU:
  Train: 100,000
  Validation: 49,369
  Overlapping row_ids: 0


## Tokenizer Configurations

In [14]:
TOKENIZER_CONFIGS = {
    "bpe_whitespace": {
        "model": "BPE",
        "pre_tokenizer": "Whitespace",
        "vocab_size": 32_000,
        "special_tokens": ["[PAD]", "[UNK]"],
        "seed": 17
    },
    "bpe_bytelevel": {
        "model": "BPE",
        "pre_tokenizer": "ByteLevel",
        "vocab_size": 32_000,
        "special_tokens": ["[PAD]", "[UNK]"],
        "seed": 17
    }
}

for name, config in TOKENIZER_CONFIGS.items():
    print(name, "->", config)

bpe_whitespace -> {'model': 'BPE', 'pre_tokenizer': 'Whitespace', 'vocab_size': 32000, 'special_tokens': ['[PAD]', '[UNK]'], 'seed': 17}
bpe_bytelevel -> {'model': 'BPE', 'pre_tokenizer': 'ByteLevel', 'vocab_size': 32000, 'special_tokens': ['[PAD]', '[UNK]'], 'seed': 17}


BPE tokenizer

In [32]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace, ByteLevel
from tokenizers.trainers import BpeTrainer

def build_tokenizer(config):
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    pre_tokenizer = config["pre_tokenizer"]

    if pre_tokenizer == "Whitespace":
        tokenizer.pre_tokenizer = Whitespace()
        alphabet = []
    elif pre_tokenizer == "ByteLevel":
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
        tokenizer.post_processor = ByteLevelProcessor(
            trim_offsets=True,
            add_prefix_space=False
        )
        alphabet = ByteLevel.alphabet()
    else:
        raise ValueError(f"Unknown pre-tokenizer: {pre_tokenizer}")

    trainer = BpeTrainer(
        vocab_size=config["vocab_size"],
        special_tokens=config["special_tokens"],
        initial_alphabet=alphabet
    )
    return tokenizer, trainer

In [18]:
for name, config in TOKENIZER_CONFIGS.items():
    tokenizer, trainer = build_tokenizer(config)
    print(name)
    print("  Model:", type(tokenizer.model).__name__)
    print("  Pre-tokenizer:", tokenizer.pre_tokenizer)
    print("  Target vocab size:", config["vocab_size"])

bpe_whitespace
  Model: BPE
  Pre-tokenizer: Whitespace()
  Target vocab size: 32000
bpe_bytelevel
  Model: BPE
  Pre-tokenizer: ByteLevel(add_prefix_space=False, trim_offsets=True, use_regex=True)
  Target vocab size: 32000


In [19]:
def train_text_iterator(splits):
    for split_name in ("uk_train", "ru_train"):
        for text in splits[split_name]["text"]:
            yield str(text)

for i, text in enumerate(train_text_iterator(splits)):
    print(repr(text[:120]))
    if i == 2:
        break

'Чим довше мають скачки тиску гіпертензією, тим сильніше змінюються стінки цих судин.'
'А поки що починали цвісти троянди, випускники складали іспити, учні відпочивали в пришкільному таборі, а директор Володи'
'Крім того, в 2020 р. багато експертів прогнозують виникнення кризових явищ на світових фінансових ринках, що може усклад'


Checking for empty values for correct `str(text)` results

In [20]:
for name in ("uk_train", "ru_train"):
    texts = splits[name]["text"]
    missing = texts.isna().sum()
    empty = texts.fillna("").astype(str).str.strip().eq("").sum()

    print(f"{name}:")
    print(f"  Total: {len(texts):,}")
    print(f"  Missing: {missing:,}")
    print(f"  Empty or whitespace-only: {empty:,}")

uk_train:
  Total: 805,983
  Missing: 0
  Empty or whitespace-only: 0
ru_train:
  Total: 100,000
  Missing: 0
  Empty or whitespace-only: 0


Training `bpe_whitespace`:

In [24]:
config = TOKENIZER_CONFIGS["bpe_whitespace"]
tokenizer_whitespace, trainer = build_tokenizer(config)

n_texts = len(splits["uk_train"]) + len(splits["ru_train"])
tokenizer_whitespace.train_from_iterator(
    train_text_iterator(splits),
    trainer=trainer,
    length=n_texts
)

print("Tokenizer:", "bpe_whitespace")
print("Vocabulary size:", tokenizer_whitespace.get_vocab_size())
print("PAD ID:", tokenizer_whitespace.token_to_id("[PAD]"))
print("UNK ID:", tokenizer_whitespace.token_to_id("[UNK]"))

Tokenizer: bpe_whitespace
Vocabulary size: 32000
PAD ID: 0
UNK ID: 1


Checking its work

In [25]:
df = splits["uk_train"]
row = df.loc[df["loc_markers"] != "[]"].iloc[1]

text = row["text"]
spans = parse_spans(row["loc_markers"])
encoding_whitespace = tokenizer_whitespace.encode(text, add_special_tokens=False)

print("Text:", text)
print("Locations:", [text[start:end] for start, end in spans])
print("\nBPE tokens:")

for token, token_id, (start, end) in zip(encoding_whitespace.tokens, encoding_whitespace.ids, encoding_whitespace.offsets):
    print(f"{token:15} ID={token_id:5} [{start}, {end})")

Text: Нагадаємо, президент України Володимир Зеленський скликав позачергове засідання Верховної Ради на 17 березня.
Locations: ['України']

BPE tokens:
Нагадаємо       ID= 1204 [0, 9)
,               ID=   13 [9, 10)
президент       ID= 1284 [11, 20)
України         ID=  619 [21, 28)
Володимир       ID= 1945 [29, 38)
Зеленський      ID= 2212 [39, 49)
склика          ID= 7771 [50, 56)
в               ID=  252 [56, 57)
позачергове     ID=31974 [58, 69)
засідання       ID= 3013 [70, 79)
Верховної       ID= 3165 [80, 89)
Ради            ID= 2131 [90, 94)
на              ID=  414 [95, 97)
17              ID= 1459 [98, 100)
березня         ID= 2352 [101, 108)
.               ID=   15 [108, 109)


Training `bpe_bytelevel`

In [26]:
config = TOKENIZER_CONFIGS["bpe_bytelevel"]
tokenizer_bytelevel, trainer = build_tokenizer(config)

n_texts = len(splits["uk_train"]) + len(splits["ru_train"])
tokenizer_bytelevel.train_from_iterator(
    train_text_iterator(splits),
    trainer=trainer,
    length=n_texts
)

print("Tokenizer: bpe_bytelevel")
print("Vocabulary size:", tokenizer_bytelevel.get_vocab_size())
print("PAD ID:", tokenizer_bytelevel.token_to_id("[PAD]"))
print("UNK ID:", tokenizer_bytelevel.token_to_id("[UNK]"))

Tokenizer: bpe_bytelevel
Vocabulary size: 32000
PAD ID: 0
UNK ID: 1


In [27]:
encoding_bytelevel = tokenizer_bytelevel.encode(text, add_special_tokens=False)

print("Text:", text)
print("Locations:", [text[start:end] for start, end in spans])
print("\nBPE tokens:")

for token, token_id, (start, end) in zip(encoding_bytelevel.tokens, encoding_bytelevel.ids, encoding_bytelevel.offsets):
    print(f"{token:15} ID={token_id:5} [{start}, {end})")

Text: Нагадаємо, президент України Володимир Зеленський скликав позачергове засідання Верховної Ради на 17 березня.
Locations: ['України']

BPE tokens:
ÐĿÐ°Ð³Ð°Ð´Ð°ÑĶÐ¼Ð¾ ID= 1326 [0, 9)
,               ID=   13 [9, 10)
ĠÐ¿ÑĢÐµÐ·Ð¸Ð´ÐµÐ½ÑĤ ID=  859 [10, 20)
ĠÐ£ÐºÑĢÐ°ÑĹÐ½Ð¸ ID=  604 [20, 28)
ĠÐĴÐ¾Ð»Ð¾Ð´Ð¸Ð¼Ð¸ÑĢ ID= 1574 [28, 38)
ĠÐĹÐµÐ»ÐµÐ½ÑģÑĮÐºÐ¸Ð¹ ID= 2827 [38, 49)
ĠÑģÐºÐ»Ð¸Ðº     ID= 8292 [49, 55)
Ð°Ð²            ID=  312 [55, 57)
ĠÐ¿Ð¾Ð·Ð°ÑĩÐµÑĢÐ³ ID=10070 [57, 66)
Ð¾Ð²Ðµ          ID= 2273 [66, 69)
ĠÐ·Ð°ÑģÑĸÐ´Ð°Ð½Ð½Ñı ID= 3283 [69, 79)
ĠÐĴÐµÑĢÑħÐ¾Ð²Ð½Ð¾ÑĹ ID= 3465 [79, 89)
ĠÐłÐ°Ð´Ð¸       ID= 2472 [89, 94)
ĠÐ½Ð°           ID=  331 [94, 97)
Ġ17             ID= 1882 [97, 100)
ĠÐ±ÐµÑĢÐµÐ·Ð½Ñı ID= 2555 [100, 108)
.               ID=   15 [108, 109)


## Tokenizer Evaluation

Both BPE tokenizers are trained on the same training corpus.

Evaluation uses fixed samples from `uk_train` and `ru_train` to analyze tokenization, alignment, sequence lengths and unknown tokens.

Validation is reserved for comparing finalized preprocessing choices and evaluating NER models.

In [51]:
tokenizers = {
    "bpe_whitespace": tokenizer_whitespace,
    "bpe_bytelevel": tokenizer_bytelevel
}

TOKENIZER_EVAL_SEED = 17
TOKENIZER_EVAL_SIZE = 5000

tokenizer_eval = {
    name: splits[name].sample(
        n=min(TOKENIZER_EVAL_SIZE, len(splits[name])),
        random_state=TOKENIZER_EVAL_SEED
    )
    for name in ("uk_train", "ru_train")
}

for name, df in tokenizer_eval.items():
    print(f"{name}: {len(df):,} texts")

uk_train: 5,000 texts
ru_train: 5,000 texts


Trying to fix byte level tokenization offsets with post-processor

In [36]:
def check_span_alignment(encoding, spans):
    results = []

    for start, end in spans:
        overlapping = [
            (a, b) for a, b in encoding.offsets
            if a < end and b > start
        ]

        if not overlapping:
            results.append("missing")
        elif overlapping[0][0] == start and overlapping[-1][1] == end:
            results.append("exact")
        else:
            results.append("partial")

    return results

In [52]:
from collections import Counter

def evaluate_alignment(tokenizer, df):
    counts = Counter()

    for row in df.itertuples(index=False):
        spans = parse_spans(row.loc_markers)
        if not spans:
            continue

        encoding = tokenizer.encode(row.text, add_special_tokens=False)
        counts.update(check_span_alignment(encoding, spans))

    return counts

In [53]:
for split_name, df in tokenizer_eval.items():
    print(f"\n{split_name}")

    for name, candidate_tokenizer in tokenizers.items():
        counts = evaluate_alignment(candidate_tokenizer, df)
        total = sum(counts.values())
        exact_pct = 100 * counts["exact"] / total if total else 0

        print(f"{name}: {dict(counts)} | Exact: {exact_pct:.2f}%")


uk_train
bpe_whitespace: {'exact': 1724, 'partial': 8} | Exact: 99.54%
bpe_bytelevel: {'exact': 1724, 'partial': 8} | Exact: 99.54%

ru_train
bpe_whitespace: {'exact': 2229, 'partial': 3} | Exact: 99.87%
bpe_bytelevel: {'exact': 2227, 'partial': 5} | Exact: 99.78%


Inspecting partial alignments

In [55]:
def inspect_partial_alignment(tokenizer, df):
    for row in df.itertuples(index=False):
        spans = parse_spans(row.loc_markers)
        if not spans:
            continue

        encoding = tokenizer.encode(row.text, add_special_tokens=False)
        results = check_span_alignment(encoding, spans)

        for (start, end), result in zip(spans, results):
            if result != "partial":
                continue

            overlapping = [
                (token, a, b)
                for token, (a, b) in zip(encoding.tokens, encoding.offsets)
                if a < end and b > start
            ]

            print("Location:", repr(row.text[start:end]))
            print("Context:", repr(row.text[max(0, start-30):end+30]))
            print("Annotation:", (start, end))
            print("Tokens:", overlapping)
            print("-" * 60)

In [56]:
for split_name, df in tokenizer_eval.items():
    for name, candidate_tokenizer in tokenizers.items():
        print(f"\n{split_name} | {name}")
        inspect_partial_alignment(candidate_tokenizer, df)


uk_train | bpe_whitespace
Location: 'Івано-Франківсь'
Context: 'Батальйон “Івано-Франківськ” отримав БТР від громади Заб'
Annotation: (11, 26)
Tokens: [('Івано', 11, 16), ('-', 16, 17), ('Франківськ', 17, 27)]
------------------------------------------------------------
Location: 'Київських гір і долин ріки Д'
Context: 'овідника "Історичний ландшафт Київських гір і долин ріки Дніпро".'
Annotation: (73, 101)
Tokens: [('Київ', 73, 77), ('ських', 77, 82), ('гір', 83, 86), ('і', 87, 88), ('до', 89, 91), ('лин', 91, 94), ('рі', 95, 97), ('ки', 97, 99), ('Дніпро', 100, 106)]
------------------------------------------------------------
Location: '”'
Context: 'рпат” на стадіоні “Арена Львів”.'
Annotation: (99, 100)
Tokens: [('”.', 99, 101)]
------------------------------------------------------------
Location: '«Б'
Context: 'виставкою у Фортечній галереї «Бастіон».'
Annotation: (124, 126)
Tokens: [('«', 124, 125), ('Ба', 125, 127)]
------------------------------------------------------------


Ukrainian texts
- All 8 problem annotations are the same for both tokenizers

russian texts:
- for 3 common problems the annotations look correct, but both tokenizers combine the closing bracket with the dot
- ByteLevel created a token that crosses the beginning of the annotation.

In [57]:
df = tokenizer_eval["ru_train"]
row = df[df["text"].str.contains("НаУкраину", regex=False)].iloc[0]

encoding = tokenizer_bytelevel.encode(row["text"], add_special_tokens=False)

print("Text:", repr(row["text"]))
print("Spans:", parse_spans(row["loc_markers"]))

for token, (start, end) in zip(encoding.tokens, encoding.offsets):
    if start < 15 and end > 0:
        print(repr(token), (start, end), repr(row["text"][start:end]))

Text: 'НаУкраину Виктор Янукович, поего словам, готов вернуться, когда «будут соблюдены условия его безопасности».'
Spans: [(2, 9)]
'ÐĿ' (0, 1) 'Н'
'Ð°Ð' (1, 3) 'аУ'
'£' (2, 3) 'У'
'ÐºÑĢÐ°' (3, 6) 'кра'
'Ð¸Ð½Ñĥ' (6, 9) 'ину'
'ĠÐĴÐ¸ÐºÑĤÐ¾ÑĢ' (10, 16) 'Виктор'


Inspecting token number per geographical name:

In [58]:
def count_location_tokens(tokenizer, df):
    counts = []

    for row in df.itertuples(index=False):
        spans = parse_spans(row.loc_markers)
        if not spans:
            continue

        encoding = tokenizer.encode(row.text, add_special_tokens=False)

        for start, end in spans:
            counts.append(sum(
                a < end and b > start
                for a, b in encoding.offsets
            ))

    return counts

In [59]:
for split_name, df in tokenizer_eval.items():
    print(f"\n{split_name}")

    for name, candidate_tokenizer in tokenizers.items():
        counts = count_location_tokens(candidate_tokenizer, df)

        print(
            f"{name}: "
            f"mean={np.mean(counts):.2f}, "
            f"median={np.median(counts):.1f}, "
            f"p95={np.percentile(counts, 95):.1f}, "
            f"max={max(counts)}"
        )


uk_train
bpe_whitespace: mean=1.76, median=1.0, p95=4.0, max=14
bpe_bytelevel: mean=1.88, median=1.0, p95=5.0, max=15

ru_train
bpe_whitespace: mean=2.21, median=2.0, p95=5.0, max=17
bpe_bytelevel: mean=2.36, median=2.0, p95=6.0, max=17


ByteLevel creates approximately 6.8% more geographic name tokens in both language samples.

Inspecting text length differences:

In [60]:
def count_sequence_lengths(tokenizer, df):
    return [
        len(tokenizer.encode(text, add_special_tokens=False).ids)
        for text in df["text"]
    ]

for split_name, df in tokenizer_eval.items():
    print(f"\n{split_name}")

    for name, candidate_tokenizer in tokenizers.items():
        lengths = count_sequence_lengths(candidate_tokenizer, df)

        print(
            f"{name}: "
            f"mean={np.mean(lengths):.2f}, "
            f"median={np.median(lengths):.1f}, "
            f"p95={np.percentile(lengths, 95):.1f}, "
            f"max={max(lengths)}"
        )


uk_train
bpe_whitespace: mean=21.77, median=19.0, p95=42.0, max=70
bpe_bytelevel: mean=22.42, median=20.0, p95=43.0, max=75

ru_train
bpe_whitespace: mean=29.19, median=27.0, p95=55.0, max=206
bpe_bytelevel: mean=30.36, median=28.0, p95=58.0, max=218


The results confirm the previous observation:
Whitespace BPE produces slightly shorter sequences on both train samples.

Checking [UNK]

In [61]:
def count_unknown_tokens(tokenizer, df):
    unk_id = tokenizer.token_to_id("[UNK]")
    total_tokens = 0
    unknown_tokens = 0

    for text in df["text"]:
        encoding = tokenizer.encode(text, add_special_tokens=False)
        total_tokens += len(encoding.ids)
        unknown_tokens += encoding.ids.count(unk_id)

    return unknown_tokens, total_tokens

In [62]:
for split_name, df in tokenizer_eval.items():
    print(f"\n{split_name}")

    for name, candidate_tokenizer in tokenizers.items():
        unknown, total = count_unknown_tokens(candidate_tokenizer, df)
        unk_pct = 100 * unknown / total if total else 0

        print(f"{name}: UNK={unknown}/{total} ({unk_pct:.4f}%)")


uk_train
bpe_whitespace: UNK=0/108864 (0.0000%)
bpe_bytelevel: UNK=0/112124 (0.0000%)

ru_train
bpe_whitespace: UNK=0/145974 (0.0000%)
bpe_bytelevel: UNK=0/151815 (0.0000%)


Checking on test set

In [63]:
test_df = pd.read_csv(data_dir / "test.csv")

for name, candidate_tokenizer in tokenizers.items():
    unknown, total = count_unknown_tokens(candidate_tokenizer, test_df)
    unk_pct = 100 * unknown / total if total else 0
    print(f"{name}: UNK={unknown}/{total} ({unk_pct:.4f}%)")

bpe_whitespace: UNK=861/42740 (2.0145%)
bpe_bytelevel: UNK=0/51013 (0.0000%)


**Tokenizer configuration conclusions:**

On the train samples Whitespace was slightly more compact. But on Kaggle test ByteLevel, BPE has an important advantage: zero unknown tokens.

## Converting spans to BIO labels

Converts character-level LOC spans into token-level BIO labels (`B-LOC`, `I-LOC`, `O`). Texts with partial alignment between entity boundaries and token offsets are skipped to avoid introducing incorrect training labels.

In [48]:
def spans_to_bio(offsets, spans):
    labels = ["O"] * len(offsets)

    for start, end in spans:
        indices = [
            i for i, (a, b) in enumerate(offsets)
            if a < end and b > start
        ]

        if not indices:
            raise ValueError(f"No tokens for span {(start, end)}")

        if offsets[indices[0]][0] != start or offsets[indices[-1]][1] != end:
            raise ValueError(f"Partial alignment for span {(start, end)}")

        labels[indices[0]] = "B-LOC"
        for i in indices[1:]:
            labels[i] = "I-LOC"

    return labels

In [49]:
encoding = tokenizer_whitespace.encode(text, add_special_tokens=False)
labels = spans_to_bio(encoding.offsets, spans)

for token, label in zip(encoding.tokens, labels):
    print(f"{token:20} {label}")

Нагадаємо            O
,                    O
президент            O
України              B-LOC
Володимир            O
Зеленський           O
склика               O
в                    O
позачергове          O
засідання            O
Верховної            O
Ради                 O
на                   O
17                   O
березня              O
.                    O


In [50]:
for row in splits["uk_val"].itertuples(index=False):
    spans = parse_spans(row.loc_markers)
    multiword_spans = [
        (start, end) for start, end in spans
        if len(row.text[start:end].split()) > 1
    ]
    if multiword_spans:
        break

print("Text:", row.text)
print("Locations:", [row.text[start:end] for start, end in spans])

for name, candidate_tokenizer in tokenizers.items():
    encoding = candidate_tokenizer.encode(row.text, add_special_tokens=False)
    labels = spans_to_bio(encoding.offsets, spans)

    print(f"\n{name}")
    for token, label, (start, end) in zip(encoding.tokens, labels, encoding.offsets):
        if label != "O":
            print(f"{repr(row.text[start:end]):20} {label}")

Text: У Львові 34-річний мешканець Яворівського району під час конфлікту жорстоко побив жінку, від спричинених травм вона померла на місці події.
Locations: ['Львові', 'Яворівського району']

bpe_whitespace
'Львові'             B-LOC
'Яворів'             B-LOC
'ського'             I-LOC
'раи'                I-LOC
'̆'                  I-LOC
'ону'                I-LOC

bpe_bytelevel
'Львові'             B-LOC
'Явор'               B-LOC
'івського'           I-LOC
'ра'                 I-LOC
'и'                  I-LOC
'̆'                  I-LOC
'ону'                I-LOC


Classifying errors on spans

In [66]:
for split_name in ("uk_train", "ru_train"):
    df = splits[split_name]
    print(f"\n{split_name}")

    for name, candidate_tokenizer in tokenizers.items():
        counts = evaluate_alignment(candidate_tokenizer, df)
        total = sum(counts.values())

        print(
            f"{name}: "
            f"exact={counts['exact']:,}, "
            f"partial={counts['partial']:,}, "
            f"missing={counts['missing']:,}, "
            f"total={total:,}"
        )


uk_train
bpe_whitespace: exact=273,723, partial=1,639, missing=0, total=275,362
bpe_bytelevel: exact=273,739, partial=1,623, missing=0, total=275,362

ru_train
bpe_whitespace: exact=46,051, partial=93, missing=0, total=46,144
bpe_bytelevel: exact=46,032, partial=112, missing=0, total=46,144


Overlapping spans

In [67]:
def count_overlapping_spans(df):
    texts_with_overlap = 0
    examples = []

    for row in df.itertuples(index=False):
        spans = sorted(parse_spans(row.loc_markers))

        if any(start2 < end1 for (_, end1), (start2, _) in zip(spans, spans[1:])):
            texts_with_overlap += 1

            if len(examples) < 3:
                examples.append((row.text, spans))

    return texts_with_overlap, examples

for split_name in ("uk_train", "ru_train"):
    count, examples = count_overlapping_spans(splits[split_name])
    print(f"\n{split_name}: {count} texts with overlapping spans")

    for text, spans in examples:
        print("Text:", repr(text[:200]))
        print("Spans:", spans)


uk_train: 0 texts with overlapping spans

ru_train: 0 texts with overlapping spans


In [68]:
def count_exact_with_overlapping_offsets(tokenizer, df):
    counts = Counter()

    for row in df.itertuples(index=False):
        spans = parse_spans(row.loc_markers)
        if not spans:
            continue

        encoding = tokenizer.encode(row.text, add_special_tokens=False)

        for (start, end), result in zip(
            spans, check_span_alignment(encoding, spans)
        ):
            if result != "exact":
                continue

            offsets = [
                (a, b) for a, b in encoding.offsets
                if a < end and b > start
            ]

            if any(a2 < b1 for (_, b1), (a2, _) in zip(offsets, offsets[1:])):
                counts["overlap"] += 1
            else:
                counts["no_overlap"] += 1

    return counts

for split_name in ("uk_train", "ru_train"):
    print(f"\n{split_name}")

    for name, candidate_tokenizer in tokenizers.items():
        counts = count_exact_with_overlapping_offsets(
            candidate_tokenizer, splits[split_name]
        )
        print(f"{name}: {dict(counts)}")


uk_train
bpe_whitespace: {'no_overlap': 273723}
bpe_bytelevel: {'no_overlap': 273687, 'overlap': 52}

ru_train
bpe_whitespace: {'no_overlap': 46051}
bpe_bytelevel: {'no_overlap': 46028, 'overlap': 4}


## BIO Data Preparation

Prepares token IDs, character offsets and BIO labels for NER training. Each record retains its original `text_id`, while `skipped_ids` tracks texts excluded due to alignment errors. The same function supports different tokenizers.

In [71]:
def prepare_bio_data(tokenizer, df, source):
    records = []
    skipped_ids = []

    for row_id, row in df.iterrows():
        spans = parse_spans(row["loc_markers"])
        encoding = tokenizer.encode(row["text"], add_special_tokens=False)

        try:
            labels = spans_to_bio(encoding.offsets, spans)
        except ValueError:
            skipped_ids.append(f"{source}_{row_id}")
            continue

        records.append({
            "text_id": f"{source}_{row_id}",
            "input_ids": encoding.ids,
            "offsets": encoding.offsets,
            "labels": labels
        })

    return records, skipped_ids

Full train BIO preparation, Whitespace BPE

In [76]:
uk_bio_whitespace, uk_skipped_whitespace = prepare_bio_data(
    tokenizer_whitespace,
    splits["uk_train"],
    source="uk"
)

print("Prepared:", len(uk_bio_whitespace))
print("Skipped:", len(uk_skipped_whitespace))
print("Total:", len(uk_bio_whitespace) + len(uk_skipped_whitespace))

Prepared: 804367
Skipped: 1616
Total: 805983


In [77]:
ru_bio_whitespace, ru_skipped_whitespace = prepare_bio_data(
    tokenizer_whitespace,
    splits["ru_train"],
    source="ru"
)

print("Prepared:", len(ru_bio_whitespace))
print("Skipped:", len(ru_skipped_whitespace))
print("Total:", len(ru_bio_whitespace) + len(ru_skipped_whitespace))

Prepared: 99909
Skipped: 91
Total: 100000


## UNO reverse - BIO to spans

Converts token-level BIO predictions back into character-level location spans using tokenizer offsets. This function is used during inference and validation to recover location boundaries in the original text.

In [78]:
def bio_to_spans(labels, offsets):
    if len(labels) != len(offsets):
        raise ValueError("Labels and offsets must have equal lengths")

    spans = []
    start, end = None, None

    for label, (a, b) in zip(labels, offsets):
        if label == "B-LOC" or (label == "I-LOC" and start is None):
            if start is not None:
                spans.append((start, end))
            start, end = a, b

        elif label == "I-LOC":
            end = max(end, b)

        elif label == "O":
            if start is not None:
                spans.append((start, end))
                start, end = None, None

        else:
            raise ValueError(f"Unknown BIO label: {label}")

    if start is not None:
        spans.append((start, end))

    return spans

Full check

In [83]:
results = []

for tokenizer_name, tokenizer in tokenizers.items():
    for split_name in ("uk_train", "ru_train"):
        df = splits[split_name]
        checked = skipped = mismatches = 0
        mismatch_ids = []

        for row_id, row in df.iterrows():
            spans = parse_spans(row["loc_markers"])
            encoding = tokenizer.encode(row["text"], add_special_tokens=False)

            try:
                labels = spans_to_bio(encoding.offsets, spans)
            except ValueError:
                skipped += 1
                continue

            restored = bio_to_spans(labels, encoding.offsets)
            checked += 1

            if sorted(spans) != sorted(restored):
                mismatches += 1
                if len(mismatch_ids) < 5:
                    mismatch_ids.append(f"{split_name}_{row_id}")

        results.append({
            "tokenizer": tokenizer_name,
            "split": split_name,
            "checked": checked,
            "skipped": skipped,
            "mismatches": mismatches,
            "mismatch_ids": mismatch_ids
        })

roundtrip_results = pd.DataFrame(results)
display(roundtrip_results)

,tokenizer,split,checked,skipped,mismatches,mismatch_ids
0,bpe_whitespace,uk_train,804367,1616,0,[]
1,bpe_whitespace,ru_train,99909,91,0,[]
2,bpe_bytelevel,uk_train,804381,1602,0,[]
3,bpe_bytelevel,ru_train,99890,110,0,[]


Lost nothing during reverse conversion of BIO → spans (however, some of the texts were missed at the stage of spans → BIO)

On Validation

In [84]:
results_val = []

for tokenizer_name, tokenizer in tokenizers.items():
    for split_name in ("uk_val", "ru_val"):
        df = splits[split_name]
        checked = skipped = mismatches = 0
        mismatch_ids = []

        for row_id, row in df.iterrows():
            spans = parse_spans(row["loc_markers"])
            encoding = tokenizer.encode(row["text"], add_special_tokens=False)

            try:
                labels = spans_to_bio(encoding.offsets, spans)
            except ValueError:
                skipped += 1
                continue

            restored = bio_to_spans(labels, encoding.offsets)
            checked += 1

            if sorted(spans) != sorted(restored):
                mismatches += 1
                if len(mismatch_ids) < 5:
                    mismatch_ids.append(f"{split_name}_{row_id}")

        results_val.append({
            "tokenizer": tokenizer_name,
            "split": split_name,
            "total": len(df),
            "checked": checked,
            "skipped": skipped,
            "mismatches": mismatches,
            "mismatch_ids": mismatch_ids
        })

roundtrip_val_results = pd.DataFrame(results_val)
display(roundtrip_val_results)

,tokenizer,split,total,checked,skipped,mismatches,mismatch_ids
0,bpe_whitespace,uk_val,201498,201123,375,0,[]
1,bpe_whitespace,ru_val,49369,49331,38,0,[]
2,bpe_bytelevel,uk_val,201498,201137,361,0,[]
3,bpe_bytelevel,ru_val,49369,49325,44,0,[]
